# Setup

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# read files from google drive
import os
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/Stanford_DeepRL_Spring2026/hw2'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)

# import modules from google drive
import sys
sys.path.append(GOOGLE_DRIVE_PATH)

Mounted at /content/drive


In [ ]:
%cd $GOOGLE_DRIVE_PATH

/content/drive/My Drive/Colab Notebooks/Stanford_DeepRL_Spring2026/hw2


# Problem 1

We implement a Q-learning agent on a 2D grid world with discrete states and discrete actions. The grid world has two goals at different distances from the start. The grid is 5 × 4 (i.e. there are 20 states), with the agent always starting at $(0, 0)$, **Goal 2** at $(4, 0)$ (4 steps away), and **Goal 1** at $(4, 3)$ (7 steps away). At each step the agent chooses from four actions: $\{left, right, up, down\}$, which move it one step on the grid. If the agent attempts to move off the edge, the agent stays in place.

## Algorithm: Tabular Q-Learning
Instead of using a neural network to represent the Q-function, the agent maintains a table of Q-values $Q(s, a)$ initialized to zero, with one Q-value for each state and action (i.e. the table contains 20 × 4 = 80 scalar Q-values). When making updates, instead of updating the parameters of a neural network, we update the Q-values in the table directly following this Q-learning update:
$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$
where $\alpha$ is the learning rate and $\gamma$ is the discount factor.

## Your Tasks
1. **Implement `choose_action` in `grid_world_q_learning.py`.**  
Complete the `choose_action` function. With probability $\epsilon$ the agent selects a uniformly random action; otherwise it selects the greedy action $a^{\ast} = arg\max_a Q(s, a)$. Break ties arbitrarily.

2. **Implement `train_q_learning` in `grid_world_q_learning.py`.**  
Complete the inner loop of `train_q_learning`. For each step in an episode:  
(a) Choose an action using $\epsilon$-greedy policy.  
(b) Step the environment to obtain $(s_{t+1}, r_t, done)$.  
(c) Apply the Q-learning update above.  
(d) Terminate the episode if `done` is `True`.  

3. **Three Scenarios.**  
Run `python gridworld_q_learning.py` on each scenario below and report the learned trajectory and total reward.  
    * **Scenario 1** $(r_{step} = -1, R_1 = 10, R_2 = 5)$
    * **Scenario 2** $(r_{step} = -2, R_1 = 10, R_2 = 5)$
    * **Scenario 3** $(r_{step} = +1, R_1 = 1, R_2 = 1)$

    <font color="red">**For each scenario, report whether it reaches a goal, and if so which one it reaches. Give a one sentence explanation for why it learned that trajectory in each scenario.**</font>

In [ ]:
!python gridworld_q_learning.py

scenario_1
  expected: goal_1
  observed: goal_1
  trajectory: ((0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2), (4, 2), (4, 3))
  actions: ('right', 'up', 'right', 'right', 'up', 'right', 'up')
  total_reward: 3.00
  start_q_values: (np.float64(1.219), np.float64(2.265), np.float64(1.219), np.float64(2.265))
scenario_2
  expected: goal_2
  observed: goal_2
  trajectory: ((0, 0), (1, 0), (2, 0), (3, 0), (4, 0))
  actions: ('right', 'right', 'right', 'right')
  total_reward: -3.00
  start_q_values: (np.float64(-4.996), np.float64(-3.057), np.float64(-4.996), np.float64(-4.329))
scenario_3
  expected: timeout
  observed: timeout
  trajectory: ((0, 0), (0, 0), (1, 0), (1, 0), (1, 0), (0, 0), (0, 0), (0, 0), (1, 0), (1, 1), (1, 2), (1, 1), (1, 0), (2, 0), (2, 1), (1, 1), (2, 1), (1, 1), (1, 2), (1, 1), (1, 0))
  actions: ('left', 'right', 'down', 'down', 'left', 'left', 'left', 'right', 'up', 'up', 'down', 'down', 'right', 'up', 'left', 'right', 'left', 'up', 'down', 'down')
  total_reward:

* **Scenario 1** $(r_{step} = -1, R_1 = 10, R_2 = 5)$  
Yes. The agent learns to reach Goal 1 because its higher terminal bonus ($R_1 = 10$) outweighs the larger step penalties incurred by the longer path, resulting in a higher total reward ($3.0$) than Goal 2 ($1.0$).

* **Scenario 2** $(r_{step} = -2, R_1 = 10, R_2 = 5)$  
Yes. As the step penalty becomes $-2$, the cost of the extra steps required to reach the distant Goal 1 dominates, making the shorter path to Goal 2 the optimal policy with a higher total reward ($-3.0$ vs. $-4.0$).

* **Scenario 3** $(r_{step} = +1, R_1 = 1, R_2 = 1)$  
No. Since the step reward is positive ($+1$) and the terminal bonuses are low ($R_1 = R_2 = 1$), the agent learns to avoid early termination by wandering around or bumping into walls to maximize its accumulated step rewards across the full horizon.